In [19]:
import pandas as pd
import pyarrow.dataset as ds
import seaborn as sns
import scanpy as sc
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import gcsfs

In [20]:
fs = gcsfs.GCSFileSystem()

In [2]:
gene_sets_path = '/home/cachris/ctc/projects/scBaseCamp/scRecount/scBaseCount_analysis/msigdb_gene_sets/results2'

## select the datasets with high scores for ZHONG_PFC_MAJOR_TYPES_EXCITATORY_NEURON

In [39]:

col = "ZHONG_PFC_MAJOR_TYPES_EXCITATORY_NEURON"

dataset = ds.dataset(gene_sets_path)  # your existing line

# 1) Read just the one column needed to compute the threshold
col_table = dataset.to_table(columns=[col])
arr = col_table.column(0).combine_chunks()

# 2) Compute the 0.9 quantile (returns a 1-element Array for q=[0.9])
threshold = pc.quantile(arr, q=[0.9]).to_pylist()[0]

# 3) Filter at the dataset level, then materialize
filtered = dataset.to_table(filter=ds.field(col) > threshold)

# 4) To pandas
sets = filtered.to_pandas()

sets

,source_file,AIZARANI_LIVER_C10_MVECS_1,AIZARANI_LIVER_C11_HEPATOCYTES_1,AIZARANI_LIVER_C12_NK_NKT_CELLS_4,AIZARANI_LIVER_C13_LSECS_2,AIZARANI_LIVER_C14_HEPATOCYTES_2,AIZARANI_LIVER_C17_HEPATOCYTES_3,AIZARANI_LIVER_C18_NK_NKT_CELLS_5,AIZARANI_LIVER_C1_NK_NKT_CELLS_1,AIZARANI_LIVER_C20_LSECS_3,...,ZHONG_PFC_C8_ORG_PROLIFERATING,ZHONG_PFC_C8_UNKNOWN_NEUROD2_POS_INTERNEURON,ZHONG_PFC_C9_ORG_OTHER,ZHONG_PFC_HES1_POS_C1_NPC,ZHONG_PFC_MAJOR_TYPES_ASTROCYTES,ZHONG_PFC_MAJOR_TYPES_EXCITATORY_NEURON,ZHONG_PFC_MAJOR_TYPES_INTERNEURON,ZHONG_PFC_MAJOR_TYPES_MICROGLIA,ZHONG_PFC_MAJOR_TYPES_NPCS,ZHONG_PFC_MAJOR_TYPES_OPC
AAACCCAAGAAGCCTG,ERX10019090.h5ad,6.600881,0.846525,8.272042,5.623971,-1.280423,-1.485655,7.466719,2.870649,12.367054,...,1.454754,8.275760,5.206219,-0.361353,5.882345,4.076207,4.350526,3.930892,3.490711,6.545411
AAACCCAAGACTTAAG,ERX10019090.h5ad,2.470227,-1.766261,4.323031,5.119047,-2.271712,-2.171578,3.558606,4.312523,13.363381,...,-1.109272,11.609733,3.332686,-0.351539,4.896018,4.572647,6.799244,2.097299,-0.245604,7.273902
AAACCCACAGTGACCC,ERX10019090.h5ad,1.932172,-3.120952,4.240504,3.748983,-3.278775,-3.267771,4.028290,3.079175,12.886997,...,0.206263,7.624137,1.182279,-1.073281,2.854363,3.697842,2.781067,2.052537,1.733663,5.799828
AAACCCAGTCGAGCTC,ERX10019090.h5ad,3.158685,-2.667979,5.094783,2.876369,-2.208594,-2.072650,3.447669,4.472614,9.299234,...,-0.924931,6.543498,2.425019,-0.690697,3.519107,5.022649,4.200116,0.725402,0.059816,5.056442
AAACCCATCATCAGTG,ERX10019090.h5ad,2.721297,-0.985809,8.464162,4.625615,-1.275935,-2.546024,7.636744,4.933544,11.958521,...,-0.353516,9.609844,4.439998,0.203931,4.246348,4.249772,4.488627,3.997209,2.269369,7.185760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GGAATAAAGTATGACA,SRX11071611.h5ad,3.534330,4.333690,2.815892,4.235553,4.858491,4.038091,24.143102,8.240075,2.888411,...,2.486743,0.520503,4.424497,-0.648123,7.072543,3.219326,0.758626,28.557455,2.131712,3.091669
TCGGTAAGTATAGGTA,SRX11071611.h5ad,-0.775029,0.568244,10.951472,-1.841391,0.820304,0.192188,5.369147,29.258746,-1.695904,...,3.468905,-1.028329,3.735960,0.599614,3.454025,3.235057,0.053784,14.360431,2.231420,0.786198
ATAGACCGTACACCGC,SRX11071615.h5ad,-1.113173,-0.818466,17.439909,-1.414515,-0.651192,-0.469268,5.642296,31.111999,2.244982,...,6.051299,-1.052150,1.837375,1.371893,3.430961,3.206169,-0.046862,16.464109,14.299773,0.986637
TTCTCAAAGCCCAATT,SRX11071615.h5ad,1.432198,-1.482990,15.840983,0.096037,-1.565404,-0.930664,6.317165,32.923961,4.301248,...,1.546899,0.369400,1.966666,0.747253,5.367673,3.454953,0.372560,17.419883,0.331675,0.303672


## Select the 20 datasets with the most exictatory neurons

In [8]:
sets['source_file'].value_counts().sort_values()

source_file
SRX11071624.h5ad        1
ERX10328897.h5ad        1
ERX10335124.h5ad        1
SRX11070674.h5ad        1
ERX10671986.h5ad        1
                    ...  
SRX10501883.h5ad    14592
SRX10501875.h5ad    16067
SRX10501881.h5ad    18725
SRX10501870.h5ad    21246
SRX10501882.h5ad    38924
Name: count, Length: 1453, dtype: int64

In [30]:
accessions = sets['source_file'].value_counts().sort_values().tail(20).index
accessions

Index(['SRX10857395.h5ad', 'SRX10857409.h5ad', 'SRX10501868.h5ad',
       'SRX10501866.h5ad', 'SRX10501891.h5ad', 'SRX10253496.h5ad',
       'SRX10501867.h5ad', 'SRX10264523.h5ad', 'SRX10264525.h5ad',
       'SRX10264526.h5ad', 'SRX10501878.h5ad', 'SRX10264524.h5ad',
       'SRX10501879.h5ad', 'SRX10501880.h5ad', 'SRX10501876.h5ad',
       'SRX10501883.h5ad', 'SRX10501875.h5ad', 'SRX10501881.h5ad',
       'SRX10501870.h5ad', 'SRX10501882.h5ad'],
      dtype='object', name='source_file')

## quick check of the study abstracts

In [31]:
import requests
import xml.etree.ElementTree as ET

def srx_to_abstract(srx):
    # 1) SRX -> find linked study/project via esearch + efetch
    esearch = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params={"db": "sra", "term": srx, "retmode": "xml"}
    )
    esearch.raise_for_status()
    root = ET.fromstring(esearch.text)
    ids = [x.text for x in root.findall(".//Id")]
    if not ids:
        return None

    # 2) Fetch the full SRA XML record
    efetch = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
        params={"db": "sra", "id": ids[0], "retmode": "xml"}
    )
    efetch.raise_for_status()
    sra_root = ET.fromstring(efetch.text)

    # 3) Look for an abstract-like field (often STUDY_ABSTRACT)
    abstract = sra_root.findtext(".//STUDY_ABSTRACT")
    title = sra_root.findtext(".//STUDY_TITLE")

    return {"title": title, "abstract": abstract}

#print(srx_to_abstract("SRX10501881"))
print(srx_to_abstract("SRX10857395"))

{'title': 'Single-cell dissection of the primary motor cortex in ALS and FTLD patients.', 'abstract': 'Amyotrophic lateral sclerosis (ALS) and frontotemporal lobar degeneration (FTLD) are two incurable and fatal neurodegenerative conditions. While distinct, they share many clinical, genetic, and pathological characteristics, and both show highly vulnerable layer 5 extratelencephalic-projecting cortical populations, including Betz cells in ALS2 and von Economo neurons (VENs) in FTLD.  Here, we report the first single-cell atlas of the human primary motor cortex and its changes in ALS and FTLD across ~380,000 nuclei from 64 control, sporadic and C9orf72-associated ALS and FTLD individuals. We identify 46 transcriptionally distinct cellular subtypes including two Betz-cell subtypes, and observe a previously-unappreciated molecular similarity between Betz cells and VENs of the frontal insula. Most dysregulated genes and pathways were shared, including stress response, ribosome function, ox

## Get the datasets from the database and query

In [32]:
import os
import shutil
import tempfile
import scanpy as sc

infiles = [
    f"gs://arc-ctc-nextflow/scBaseCount-publish/prod/2026-01-12/h5ad/GeneFull_Ex50pAS/Homo_sapiens/{accession}"
    for accession in accessions
]

adata = []

with tempfile.TemporaryDirectory(prefix="h5ad_") as tmpdir:
    for infile in infiles:
        print(infile)
        local_path = os.path.join(tmpdir, os.path.basename(infile))
        with fs.open(infile, "rb") as src, open(local_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        adata.append(sc.read_h5ad(local_path))

gs://arc-ctc-nextflow/scBaseCount-publish/prod/2026-01-12/h5ad/GeneFull_Ex50pAS/Homo_sapiens/SRX10857395.h5ad
gs://arc-ctc-nextflow/scBaseCount-publish/prod/2026-01-12/h5ad/GeneFull_Ex50pAS/Homo_sapiens/SRX10857409.h5ad
gs://arc-ctc-nextflow/scBaseCount-publish/prod/2026-01-12/h5ad/GeneFull_Ex50pAS/Homo_sapiens/SRX10501868.h5ad


FileNotFoundError: b/arc-ctc-nextflow/o/scBaseCount-publish%2Fprod%2F2026-01-12%2Fh5ad%2FGeneFull_Ex50pAS%2FHomo_sapiens%2FSRX10501868.h5ad

In [35]:
infile = 'gs://arc-scbasecount/2025-02-25/h5ad/GeneFull_Ex50pAS/Homo_sapiens/SRX10501868.h5ad'
local_path = os.path.join(tmpdir, os.path.basename(infile))
with fs.open(infile, "rb") as src, open(local_path, "wb") as dst:
    shutil.copyfileobj(src, dst)
adata.append(sc.read_h5ad(local_path))

FileNotFoundError: b/arc-scbasecount/o/2025-02-25%2Fh5ad%2FGene%2FHomo_sapiens%2FSRX10501868.h5ad

In [37]:
adata = sc.concat(adata)

/home/cachris/.conda/envs/sc-env5/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [38]:
adata.obs

,gene_count_Unique,umi_count_Unique,gene_count_UniqueAndMult-EM,umi_count_UniqueAndMult-EM,gene_count_UniqueAndMult-Uniform,umi_count_UniqueAndMult-Uniform,SRX_accession,cell_type,cell_ontology_term_id
AAACCCAAGAGGCTGT,1682,2859.0,1783,2963.998779,1796,2963.998535,SRX10857395,oligodendrocyte,CL:0000128
AAACCCAAGATGATTG,1971,3465.0,2087,3566.998291,2104,3566.998291,SRX10857395,astrocyte,CL:0000127
AAACCCAAGATGTAGT,4629,11604.0,4829,11991.996094,4955,11991.993164,SRX10857395,neuron,CL:0000540
AAACCCAAGCCTGACC,2163,4140.0,2257,4306.999512,2291,4306.999023,SRX10857395,oligodendrocyte,CL:0000128
AAACCCAAGCTTTCTT,552,666.0,576,717.000000,576,717.000000,SRX10857395,oligodendrocyte,CL:0000128
...,...,...,...,...,...,...,...,...,...
TTTGTTGTCGACTCCT,4336,10891.0,4470,11177.002930,4582,11177.000000,SRX10857409,neuron,CL:0000540
TTTGTTGTCTAAGAAG,678,952.0,711,984.000061,715,984.000000,SRX10857409,oligodendrocyte,CL:0000128
TTTGTTGTCTGGACTA,710,870.0,766,914.999817,770,914.999817,SRX10857409,central nervous system macrophage,CL:0000878
TTTGTTGTCTTACGGA,1257,1910.0,1319,1961.000244,1333,1961.000244,SRX10857409,oligodendrocyte,CL:0000128


## Import gene sets

In [41]:
adata.obs['srx_code'] = adata.obs['SRX_accession'].astype(str) + '_' + adata.obs.index
sets['srx_code'] = sets['source_file'].str.split('.').str[0].astype(str) + '_' + sets.index
adata.obs = pd.merge( adata.obs, sets, on='srx_code', how = 'left')

/home/cachris/.conda/envs/sc-env5/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [42]:
adata.obs

,gene_count_Unique,umi_count_Unique,gene_count_UniqueAndMult-EM,umi_count_UniqueAndMult-EM,gene_count_UniqueAndMult-Uniform,umi_count_UniqueAndMult-Uniform,SRX_accession,cell_type,cell_ontology_term_id,srx_code,...,ZHONG_PFC_C8_ORG_PROLIFERATING,ZHONG_PFC_C8_UNKNOWN_NEUROD2_POS_INTERNEURON,ZHONG_PFC_C9_ORG_OTHER,ZHONG_PFC_HES1_POS_C1_NPC,ZHONG_PFC_MAJOR_TYPES_ASTROCYTES,ZHONG_PFC_MAJOR_TYPES_EXCITATORY_NEURON,ZHONG_PFC_MAJOR_TYPES_INTERNEURON,ZHONG_PFC_MAJOR_TYPES_MICROGLIA,ZHONG_PFC_MAJOR_TYPES_NPCS,ZHONG_PFC_MAJOR_TYPES_OPC
0,1682,2859.0,1783,2963.998779,1796,2963.998535,SRX10857395,oligodendrocyte,CL:0000128,SRX10857395_AAACCCAAGAGGCTGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1971,3465.0,2087,3566.998291,2104,3566.998291,SRX10857395,astrocyte,CL:0000127,SRX10857395_AAACCCAAGATGATTG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4629,11604.0,4829,11991.996094,4955,11991.993164,SRX10857395,neuron,CL:0000540,SRX10857395_AAACCCAAGATGTAGT,...,-1.777883,11.838266,4.165150,-0.859902,1.402472,5.051536,7.000393,-0.316780,-2.337396,6.748510
3,2163,4140.0,2257,4306.999512,2291,4306.999023,SRX10857395,oligodendrocyte,CL:0000128,SRX10857395_AAACCCAAGCCTGACC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,552,666.0,576,717.000000,576,717.000000,SRX10857395,oligodendrocyte,CL:0000128,SRX10857395_AAACCCAAGCTTTCTT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30893,4336,10891.0,4470,11177.002930,4582,11177.000000,SRX10857409,neuron,CL:0000540,SRX10857409_TTTGTTGTCGACTCCT,...,-1.175357,11.974468,5.172667,-0.843534,4.642896,5.202025,4.630666,1.807541,-2.401690,9.950539
30894,678,952.0,711,984.000061,715,984.000000,SRX10857409,oligodendrocyte,CL:0000128,SRX10857409_TTTGTTGTCTAAGAAG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30895,710,870.0,766,914.999817,770,914.999817,SRX10857409,central nervous system macrophage,CL:0000878,SRX10857409_TTTGTTGTCTGGACTA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30896,1257,1910.0,1319,1961.000244,1333,1961.000244,SRX10857409,oligodendrocyte,CL:0000128,SRX10857409_TTTGTTGTCTTACGGA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Quick plotting

In [45]:
sc.pp.normalize_per_cell(adata)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color = ['cell_type', 'ZHONG_PFC_MAJOR_TYPES_EXCITATORY_NEURON', 'SRX_accession'])

/tmp/ipykernel_257687/971632820.py:1: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(adata)


/home/cachris/.conda/envs/sc-env5/lib/python3.13/site-packages/scanpy/preprocessing/_simple.py:591: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(
